# 04 - Ablation tests

**Status: scaffold.** Sections and helper calls are sketched; the analysis is
not written.

Direct port of the ablation workflow in
`inhib_modulation/02_inhibitory_modulation_analysis.ipynb`, with the readouts
replaced by continuous-time ones. Each ablated network is summarized by
`jc.condition_summary`, which returns stability, non-normality, transient, and
DC-gain numbers in a single row, and `jc.ablation_sweep` differences every
condition against the intact network.

**Methodological note.** Ablations are *not* renormalized by default. A cut
changes the network's gain, and renormalizing afterwards would scale that change
away and report only the residual structural effect. Both questions are
legitimate but they are different questions: run the sweep twice, with
`renormalize=None` and `renormalize="spectral_radius"`, and report both.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import jacobian_core as jc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": 0.25})

MATRIX = "matrices/mij_matrix.csv"
NETLIST = "matrices/mij_netlist.csv"

# Synaptic gain. rho(W) < 1 guarantees a stable Jacobian, since eigenvalues of
# J = -I + W are those of W shifted left by one. 0.95 matches the normalization
# used in schur_decomp and inhib_modulation so results stay comparable.
GAIN = 0.95
TAU = 1.0        # membrane time constant; time is measured in units of tau
LEAK = 1.0       # coefficient on -I; leave at 1 unless testing leak sensitivity

data = jc.load_jacobian_data(MATRIX, NETLIST)
labels = data.labels
masks = jc.ei_masks(data.ei, labels)
W, norm_info = jc.normalize_weights(data.W_raw, method="spectral_radius", target=GAIN)
J = jc.build_jacobian(W, tau=TAU, leak=LEAK)
baseline = jc.stability_summary(J)
print(f"alpha = {baseline['spectral_abscissa']:.4f}   omega = {baseline['numerical_abscissa']:.4f}")

OUT = jc.output_dir("04_jacobian_ablation_tests")
n = len(labels)

## 1. Single-cell-type ablations

Silence one cell type at a time (`jc.ablate_nodes`) and rank by the change in
stability margin, numerical abscissa, and peak amplification. The three
rankings need not agree, and where they disagree is the interesting part: a cell
whose removal barely moves the eigenvalues but collapses the transient is doing
structural, not spectral, work.

In [ ]:
# TODO:
# conditions = {labels[i]: jc.ablate_nodes(W, [i]) for i in range(n)}
# table = jc.ablation_sweep(W, conditions, reference="intact", with_transient=True)

## 2. Edge ablations

Rank individual connections. Start with the strongest edges by weight, then by
the eigenvalue-sensitivity criterion $|v_i u_j|$ from the dominant left and right
eigenvectors, which predicts first-order spectral effect without re-solving.

In [ ]:
# TODO: top-k edges by weight and by sensitivity; jc.ablate_edges; compare predicted vs actual shift.

## 3. Graded weakening

Ablation is the endpoint of a continuum. Sweep `factor` from 1.0 to 0.0 for the
top candidates and check whether the effect is linear or has a threshold.

In [ ]:
# TODO: factor sweep per candidate; plot stability margin and peak amplification vs factor.

## 4. Null comparison

Compare measured ablation effects against a block-preserving weight shuffle, so
"this edge matters" is a claim about this network rather than about any network
with this degree and weight distribution.

In [ ]:
# TODO: shuffled nulls; z-scores per condition.

## 5. Save and verify

In [ ]:
# TODO: save both renormalized and un-renormalized sweeps; assert the intact row matches notebook 01.